In [ ]:
import torch
import sys
import subprocess


def run_pip(command):
    # 공백 기준으로 명령어를 분리하여 리스트로 만듭니다.
    cmd = [sys.executable, "-m", "pip", "install"] + command.split()
    print(f"🔄 실행 중: {' '.join(cmd)}")
    subprocess.check_call(cmd)


print(
    f"🔥 현재 환경 감지 중... Python {sys.version.split()[0]} / PyTorch {torch.__version__}"
)

# 1. PyTorch 2.4.x 환경인지 확인
if "2.4" in torch.__version__:
    print("✅ 최신 PyTorch 2.4.0 환경입니다. H200 최적화 설치를 진행합니다.")

    # 2. pip 업그레이드 & 빌드 도구 설치
    run_pip("--upgrade pip")
    run_pip("packaging ninja")

    # 3. Flash Attention 2 '완제품' 강제 설치
    print("⬇️ Flash Attention 2 (Pre-built Wheel) 다운로드 중...")
    run_pip(
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
    )

    # 4. Unsloth 설치 (공백 제거!)
    print("⬇️ Unsloth (H200 Optimized) 설치 중...")
    # [수정됨] @ 앞뒤의 공백을 제거하여 하나의 문자열로 인식하게 함
    run_pip(
        "unsloth[cu124-ampere-torch240]@git+https://github.com/unslothai/unsloth.git"
    )

    # 5. 나머지 의존성 설치
    print("⬇️ 기타 라이브러리 설치 중...")
    run_pip(
        "trl peft accelerate bitsandbytes huggingface_hub python-dotenv pandas openai matplotlib seaborn scipy"
    )

    print("\n🎉 설치가 완벽하게 끝났습니다!")
    print("👉 상단 메뉴의 [Kernel] -> [Restart Kernel]을 누르고 프로젝트를 시작하세요.")

else:
    print(
        f"⚠️ 경고: 선택한 이미지가 PyTorch 2.4.0이 아닙니다. (현재: {torch.__version__})"
    )
    print("RunPod 템플릿에서 'PyTorch 2.4.0'을 선택했는지 다시 확인해주세요.")

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

In [ ]:
# 1. 작업 폴더로 이동
cd /workspace

# 2. 볼륨 디스크에 실제 저장소 폴더 생성
mkdir -p /workspace/hf_cache
mkdir -p /workspace/torch_cache
mkdir -p /workspace/tmp

# 3. 시스템 디스크의 기본 캐시 폴더 삭제 (충돌 방지)
rm -rf /root/.cache/huggingface
rm -rf /root/.cache/torch

# 4. [핵심] 심볼릭 링크 생성 (시스템 경로 -> 볼륨 디스크로 납치)
# 이제 시스템이 /root/.cache에 저장하려 하면 자동으로 /workspace로 들어갑니다.
mkdir -p /root/.cache
ln -s /workspace/hf_cache /root/.cache/huggingface
ln -s /workspace/torch_cache /root/.cache/torch

# 5. 환경 변수 강제 설정 (현재 세션용)
export HF_HOME="/workspace/hf_cache"
export TORCH_HOME="/workspace/torch_cache"
export TMPDIR="/workspace/tmp"
export TEMP="/workspace/tmp"

echo "✅ 디스크 우회 설정 완료! 이제 모든 데이터는 /workspace에 저장됩니다."

In [ ]:
# merge.py (Disk Full 방지 버전)
from unsloth import FastLanguageModel
import os
import shutil
from dotenv import load_dotenv

# ==========================================
# 0. [필수] 디스크 경로 강제 설정 (최상단 배치)
# ==========================================
# 모든 임시 파일과 캐시를 넓은 /workspace로 돌림
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["TMPDIR"] = "/workspace/tmp"  # 임시 파일 경로 변경
os.environ["TORCH_HOME"] = "/workspace/torch_cache"

# 임시 폴더 생성
os.makedirs("/workspace/tmp", exist_ok=True)
os.makedirs("/workspace/hf_cache", exist_ok=True)

# ==========================================
# 1. 병합 로직
# ==========================================
load_dotenv()

# HF에 올린 DPO 모델 ID (수정 확인!)
ADAPTER_MODEL = "hyunNus/Woooly-SFT-DPO-70B"
MERGED_DIR = "/workspace/merged_woooly_70b"

print(f"🧹 공간 확보 및 설정 완료. 병합 시작: {ADAPTER_MODEL}")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_MODEL,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
        token=os.getenv("HUGGINGFACE_API_KEY_FOR_READ_ONLY"),
    )

    # 16bit 병합 저장
    print("💾 병합된 모델 저장 중 (약 130GB)...")
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    print(f"✅ 병합 성공! 저장 위치: {MERGED_DIR}")

except Exception as e:
    print(f"❌ 병합 실패: {e}")
    # 실패 시 임시 파일 정리 안내
    print(
        "👉 터미널에서 'df -h'로 용량을 확인하고, 'rm -rf /workspace/tmp'로 임시 파일을 지우세요."
    )

In [ ]:
# 1. 작업 폴더 이동 (필수)
cd /workspace

# 2. .env 파일 생성 (토큰 입력)
# (이미 있다면 패스)

# 3. 모델 병합 실행 (최초 1회만)
python merge_model.py

# 4. vLLM 및 ngrok 설치
pip install vllm pyngrok

# 5. vLLM 서버 백그라운드 실행
# H200 메모리를 90% 활용하여 최대 성능을 냄
nohup python -m vllm.entrypoints.openai.api_server \
    --model /workspace/merged_woooly_70b \
    --dtype float16 \
    --gpu-memory-utilization 0.9 \
    --max-model-len 2048 \
    --port 8000 > vllm.log 2>&1 &

print("⏳ vLLM 서버가 시작되었습니다. 로그는 vllm.log를 확인하세요.")

# 6. ngrok 터널링 (외부 접속 주소 생성)
ngrok http 8000

👉 출력된 https://xxxx.ngrok-free.app 주소를 복사하세요!